In [0]:
import json
from pyspark.sql import Row
from pyspark.sql.window import Window
from pyspark.sql.functions import *

## Flattening the bronze schedule into games table

In [0]:
def extract_games_from_schedule(schedule_json: dict) -> list[dict]:
    """
    Given one day's schedule JSON, return one dict per game.
    """
    rows = []
    for day in schedule_json.get("dates", []):
        game_date = day.get("date")
        for game in day.get("games", []):
            rows.append({
                "game_pk": game.get("gamePk"),
                "game_date": game_date,
                "home_team_id": game.get("teams", {}).get("home", {}).get("team", {}).get("id"),
                "away_team_id": game.get("teams", {}).get("away", {}).get("team", {}).get("id"),
                "game_status": game.get("status", {}).get("detailedState"),
                "double_header": game.get("doubleHeader"),
            })
    return rows

In [0]:
bronze_schedule = spark.table("bronze.mlb_schedule_raw")

all_schedule_rows = []
schedule_parse_failures = []

for row in bronze_schedule.collect():
    try:
        parsed = json.loads(row.raw_json)
        all_schedule_rows.extend(extract_games_from_schedule(parsed))
    except json.JSONDecodeError as e:
        print(f"[WARN] failed to parse schedule for {row.source_date}: {e}")
        schedule_parse_failures.append(row.source_date)

print(f"All games: {len(all_schedule_rows)}")
print(f"Schedule parse failures: {len(schedule_parse_failures)}")

In [0]:
schedule_raw_df = spark.createDataFrame([Row(**r) for r in all_schedule_rows])

# Ranking duplicates: prefer "Final" over non-final duplicates, since a game's status can change across days (ex: Postponed -> Final)
status_priority = when(col("game_status") == "Final", lit(0)).otherwise(lit(1))

schedule_window = Window.partitionBy("game_pk").orderBy(status_priority)

silver_schedule = (
    schedule_raw_df
    .withColumn("status_rank", status_priority)
    .withColumn("row_num", row_number().over(schedule_window))
    .filter(col("row_num") == 1)
    .drop("row_num", "status_rank")
    .withColumn("silver_processed_at", current_timestamp())
)

In [0]:
silver_schedule.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.mlb_schedule")

In [0]:
print(f"silver.mlb_schedule rows: {silver_schedule.count()}")
display(silver_schedule.limit(10))

## Deduping Bronze schedule

## Running flattening across all games in Silver's input

### Using the deduped bronze schedule for games